In [2]:
import carla
import numpy as np
import logging
import pygame
import cv2
import random


pygame 2.6.0 (SDL 2.28.4, Python 3.12.10)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
CARLA_LOCALHOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 40.0

logging.basicConfig(
    level = logging.INFO,
    format = "%(asctime)s [%(levelname)s] %(name)s: %(message)s"
)

random.seed(42)

logger = logging.getLogger(__name__)
IMAGE_Z = 3
IMAGE_X = -6
IMAGE_Y = 0
CAMERA_INIT_TRANS = carla.Transform(carla.Location(z = IMAGE_Z, x = IMAGE_X, y = IMAGE_Y))

In [4]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_LOCALHOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()

    if not world:
        raise RuntimeError(
            "Carla was not able to return a valid world."
        )

    logger.info(
        "Connection to client established."
    )
    return client, world

In [5]:
def load_map(world: carla.World, client: carla.Client, map: str = "Town05") -> None:

    if world.get_map().name.split("/")[-1] == map:
        logger.info(f"Assigned map {map} is already loaded.")
    else:
        try:
            world = client.load_world(map)
        except:
            world = client.load_world("Town01")
        finally:
            logger.info(
                f"Map: {map} is loaded."
            )
            

In [6]:
def set_spectator(world: carla.World) -> None:

    spectator = world.get_spectator()
    spectator.set_transform(
        carla.Transform(
            carla.Location(
                x = 0,
                y = 0,
                z = 200
            ),
            carla.Rotation(
                pitch = -90,
                yaw = 0,
                roll = 0
            )
        )
    )

In [7]:
def load_actors(world: carla.World) -> None:
    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")
    spawn_points = world.get_map().get_spawn_points()
    vehicles = []

    for i in range(0, 10, 1):
        spawn_point = random.choice(spawn_points)
        blueprint = random.choice(vehicle_bp)

        vehicle = world.try_spawn_actor(
            blueprint,
            spawn_point
        )

        if vehicle is not None:
            vehicle.set_autopilot(True)
            vehicles.append(vehicle)

In [8]:
def ego_vehicle(world: carla.World) -> carla.Vehicle:

    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")
    spawn_points = world.get_map().get_spawn_points()
    spawn_point = random.choice(spawn_points)
    if not vehicle_bp:
        raise RuntimeError(
            "No valid blueprint return to cybertruck."
        )

    vehicle = world.try_spawn_actor(
        vehicle_bp[0],
        spawn_point
    )

    return vehicle

In [9]:
def load_rgb_sensor(world: carla.World, vehicle: carla.Vehicle) -> tuple[int, int, carla.Sensor]:

    rgb_bp = world.get_blueprint_library().find("sensor.camera.rgb")

    if rgb_bp is None:
        raise RuntimeError(
            "It was not possible to retrieve a valid RGB blurprint."
        )

    rgb_bp.set_attribute('image_size_x', '600')
    rgb_bp.set_attribute('image_size_y', '600')
    
    image_h = rgb_bp.get_attribute("image_size_x").as_int()
    image_w = rgb_bp.get_attribute("image_size_y").as_int()

    rgb = world.try_spawn_actor(
        rgb_bp,
        CAMERA_INIT_TRANS,
        attach_to = vehicle
    )

    logger.info("RGB camera is loaded to the EGO vehicle.")

    return image_h, image_w, rgb
    

In [10]:
def rgb_callback(data, data_dict):

    image = np.frombuffer(
        data.raw_data,
        dtype=np.uint8
    )

    image = image.reshape(
        (data.height, data.width, 4)
    )

    # Convert BGRA to RGB
    image = image[:, :, :3][:, :, ::-1]

    data_dict["rgb_img"] = image

In [11]:
def steering_control(vehicle: carla.Vehicle, key) -> None:

    control = carla.VehicleControl()

    # Default lights
    current_lights = carla.VehicleLightState.Position

    if key[pygame.K_w]:
        control.throttle = 1.0

    if key[pygame.K_s]:
        control.brake = 0.5

        # Activate brake lights
        current_lights = carla.VehicleLightState(
            int(current_lights) | int(carla.VehicleLightState.Brake)
        )

    if key[pygame.K_d]:
        control.steer = 0.5

    if key[pygame.K_a]:
        control.steer = -0.5

    if key[pygame.K_SPACE]:
        control.hand_brake = True

    vehicle.set_light_state(current_lights)
    vehicle.apply_control(control)

In [19]:
def draw_hud(screen, vehicle):

    width = 260
    height = screen.get_height()

    # Transparent overlay
    hud = pygame.Surface(
        (width, height),
        pygame.SRCALPHA
    )

    # Colors
    WHITE = (240, 240, 240)
    GREY = (180, 180, 180)
    ORANGE = (255, 140, 0)
    DARK = (15, 15, 15, 175)

    # Fonts
    title_font = pygame.font.SysFont("Arial", 21, bold=True)
    font = pygame.font.SysFont("Arial", 16)
    small_font = pygame.font.SysFont("Arial", 13)

    # Background with alpha transparency
    hud.fill(DARK)

    # Vehicle information
    velocity = vehicle.get_velocity()
    speed = 3.6 * (
        velocity.x**2 +
        velocity.y**2 +
        velocity.z**2
    )**0.5

    location = vehicle.get_location()
    transform = vehicle.get_transform()
    control = vehicle.get_control()

    heading = transform.rotation.yaw % 360

    # Helper: draw text
    def draw_text(text, x, y, color=WHITE, font=font):
        surface = font.render(text, True, color)
        hud.blit(surface, (x, y))

    # Helper: section separator
    def separator(y):
        pygame.draw.line(
            hud, ORANGE,
            (15, y), (width - 15, y), 2
        )

    # Helper: horizontal bar
    def draw_bar(label, value, y, color=ORANGE):

        draw_text(label, 15, y, GREY, small_font)

        bar_x = 105
        bar_width = 130
        bar_height = 9

        pygame.draw.rect(
            hud, (100, 100, 100),
            (bar_x, y + 3, bar_width, bar_height)
        )

        fill_width = int(
            max(0.0, min(1.0, abs(value))) * bar_width
        )

        pygame.draw.rect(
            hud, color,
            (bar_x, y + 3, fill_width, bar_height)
        )

    # HEADER
    draw_text("VEHICLE TELEMETRY", 15, 15, WHITE, title_font)
    separator(48)

    # VEHICLE
    draw_text("Vehicle: Ego Vehicle", 15, 60, GREY)
    draw_text(vehicle.type_id.split(".")[-1], 115, 60)

    draw_text("Speed:", 15, 88, GREY)
    draw_text(f"{speed:.1f} km/h", 115, 88)

    draw_text("Heading:", 15, 116, GREY)
    draw_text(f"{heading:.1f} deg", 115, 116)

    separator(148)

    # POSITION
    draw_text("POSITION", 15, 160, WHITE, title_font)

    draw_text(f"X: {location.x:.2f} m", 15, 192)
    draw_text(f"Y: {location.y:.2f} m", 15, 218)
    draw_text(f"Z: {location.z:.2f} m", 15, 244)

    separator(278)

    # CONTROLS
    draw_text("CONTROLS", 15, 290, WHITE, title_font)

    draw_bar("Throttle", control.throttle, 330)
    draw_bar("Brake", control.brake, 360, (220, 70, 70))
    draw_bar("Steering", control.steer, 390, (70, 170, 230))

    draw_text(
        f"Reverse: {control.reverse}",
        15, 425, GREY
    )

    draw_text(
        f"Hand brake: {control.hand_brake}",
        15, 450, GREY
    )

    draw_text(
        f"Gear: {control.gear}",
        15, 475, GREY
    )

    separator(510)

    # Overlay on top of camera
    screen.blit(hud, (0, 0))


In [14]:
def main():
    client, world = connect_to_carla()
    actors = world.get_actors().filter("vehicle.*")
    for vehicle in actors:
        vehicle.destroy()
    load_map(world, client, "Town05")
    set_spectator(world)
    load_actors(world)
    vehicle = ego_vehicle(world)
    image_h, image_w, rgb = load_rgb_sensor(world, vehicle)

    data_dict = {
        "rgb_img" : None,
    }
    WINDOW_WIDTH = 1080
    WINDOW_HEIGHT = 1080
    running = True
    rgb.listen(lambda image: rgb_callback(image, data_dict))
    
    try:
        pygame.init()
        screen = pygame.display.set_mode((WINDOW_WIDTH, WINDOW_HEIGHT))
        pygame.display.set_caption("Manual Control")
        while running:

            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False
                if event.type == pygame.KEYDOWN:
                    if event.key == pygame.K_q:
                        running = False

            key = pygame.key.get_pressed()
            steering_control(vehicle, key)
            rgb_data = data_dict["rgb_img"]

            if rgb_data is not None:
                # Convert NumPy array into Pygame surface
                surface = pygame.surfarray.make_surface(
                    np.transpose(rgb_data, (1, 0, 2))
                )

                surface = pygame.transform.smoothscale(
                    surface,
                    screen.get_size()
                )

                # Display image
                screen.blit(surface, (0, 0))

            draw_hud(screen, vehicle)
            pygame.display.flip()
    finally:
        rgb.stop()
        rgb.destroy()
        vehicle.destroy()
        pygame.quit()



In [20]:
if __name__ == "__main__":
    main()

2026-11-14 12:32:29,902 [INFO] __main__: Connection to client established.
2026-11-14 12:32:30,338 [INFO] __main__: Assigned map Town05 is already loaded.
2026-11-14 12:32:31,296 [INFO] __main__: RGB camera is loaded to the EGO vehicle.
